In [ ]:
from ecmwfapi import ECMWFService
import earthkit.data
import earthkit.plots

# ECMWFAPI for Mars WebAPI retrievals

## [Web API Documentation](https://confluence.ecmwf.int/display/WEBAPI/ECMWF+Web+API+Home)

In [ ]:
server = ECMWFService("mars")

request = {
    "class": "od",
    "date": "-1",
    "expver": "1",
    "param": 140120,
    "step": "12",
    "stream": "wave",
    "time": "00",
    "type": "fc",
    "grid": "0.25/0.25", # interpolation done on client
    "area": "55/-15/34.0/5.0",  #+N, -W, -S, +E
    'format' : "netcdf" # not fully supported, wont work for all fields, only available from webapi. Larger than grib!
    # Recommended to convert with eccodes on local machine to reduce queueing times on webapi. Can also be done on mars catalogue
}
server.execute(request, "../data/target.nc")

### Check your request status [here](https://apps.ecmwf.int/webmars/joblist/), it should show up on webapi

In [ ]:
fs = earthkit.data.from_source("file", "../data/target.nc")

In [ ]:
fx = fs.to_xarray()

In [ ]:
chart = earthkit.plots.Map()
chart.contourf(
    fx
)
chart.legend()
chart.title()
chart.coastlines()
chart.gridlines()
chart.show()

### Cost of a request

MARS can estimate the cost of a request — number of fields, disk files, tape files, and total size — without actually retrieving any data, using the `list,output=cost,...` action. For cost, the request is given as a string in MARS syntax (rather than a dict).

In [ ]:
server = ECMWFService("mars")
server.execute(
    "list,"
    "output=cost,"
    "class=od,"
    "stream=oper,"
    "expver=1,"
    "type=fc,"
    "levtype=sfc,"
    "param=167.128,"
    "date=-2,"
    "time=00,"
    "step=0/to/240/by/12",
    "../data/cost.txt"
)


## Multi-target requests

MARS supports a *multi-target* syntax (`target="name_[keyword].ext"`) that automatically splits output into one file per keyword value — but only when running the MARS client directly on ECMWF's own compute. The Web API does **not** support this: it always writes a single file, literally named with the brackets included. To get the same result through the Web API, retrieve everything into one file and then split it client-side with `earthkit.data`.

In [ ]:
server = ECMWFService("mars")

request = {
    "class": "od",
    "date": "-1",
    "expver": "1",
    "param": "167.128",  # 2m temperature
    "levtype": "sfc",
    "stream": "oper",
    "time": "00",
    "type": "fc",
    "step": "0/6/12",
    "grid": "1/1",
}
# NOTE: the Web API does not interpret the bracket syntax below - it will
# literally create a single file called "multi_target_[step].grib"
server.execute(request, "../data/multi_target_[step].grib")

Check how many files were actually created:

Only one file was produced. Now split it into one file per step, client-side, using earthkit-data's `to_target("file-pattern", ...)`:

In [ ]:
ds = earthkit.data.from_source("file", "../data/multi_target_[step].grib")
ds.to_target("file-pattern", "../data/split_step_{metadata.step}.grib")

Confirm the split files now exist